---
title: "Exercise 4A: Visium HD Normalization"
format: html
editor: source
editor_options:
  chunk_output_type: console
jupyter: ir
---

## Normalization and scaling for Visium HD

In this exercise, we normalize the filtered Visium HD object from yesterday's `Exercise 2`. Standard library-size normalization, as used for scRNA-seq analysis, is a useful starting point, but spatial data introduces additional challenges: total counts can vary because of local capture efficiency, tissue density, morphology, or real biology.

## Learning objectives

By the end of this exercise, you will be able to:

- Add log-normalized values to a Visium HD `SpatialExperiment` or `SpatialFeatureExperiment` object.
- Inspect size factors and assay names.
- Scale selected features for visualization and downstream multivariate methods.
- Understand the principles of spatially-aware normalization and evaluate whether it could be useful for a given dataset.

## Libraries

In [ ]:
#| warning: false
#| output: false

library(SpatialExperiment)
library(SpatialFeatureExperiment)
library(scuttle)
library(scrapper)
library(scater)
library(HDF5Array)
library(ggspavis)
library(alabaster.sfe)
library(Voyager)

## Load the Visium HD object

We start with the filtered Visium HD object saved at the end of Exercise 2.

In [ ]:
# Reload the SpatialFeatureExperiment object if not in the R session already
sfe <- readObject("results/day1/01.2_sfe_visium/")
sfe

## Log-normalization

The object contains raw counts. We use the `scuttle::logNormCounts()` function to compute size factors and add a `logcounts` assay.

::: callout-note
This function will deprecate soon and replaced by the faster `scrapper::normalizeRnaCounts.se()` function.
:::

In [ ]:
assayNames(sfe)
sfe <- logNormCounts(sfe)

::: callout-important
## Exercise 1

What assay was added to the Visium HD object? What do the size factors represent?
:::

::: {.callout-tip collapse="true"}
## Answer

In [ ]:
assayNames(sfe)
summary(sizeFactors(sfe))

The `logcounts` assay contains log-normalized expression values. The size factors adjust for differences in total count depth between cells.

::: callout-note
Other more advanced normalization methods exist for scRNA-seq data, which aim at adjusting for compositional biases across cells, for example `scuttle::computePooledFactors()`, or at stabilizing the variance of such low-counts data, for example `Seurat:SCTranform()`
:::
:::

We can inspect whether size factors have a spatial pattern. 

In [ ]:
colData(sfe)$sizeFactor <- sizeFactors(sfe)

plotSpatialFeature(
    sfe,
    colGeometryName = "cellSeg", features = "sizeFactor"
)

::: callout-important
## Exercise 2

Do size factors show a spatial pattern across the tissue region? What biological or technical effects could explain this?
:::

## Scaling selected features

Normalization adjusts for differences in total counts between observations. Scaling is a different step: it centers and rescales features so that they can be compared on a common scale. This is often useful for multivariate methods such as PCA, clustering, or heatmaps, to avoid giving more weight to more highly expressed genes.

Here we demonstrate scaling on a small set of marker genes.

In [ ]:
marker_genes <- c("PIGR", "CEACAM5", "MUC17", "OLFM4")
marker_genes <- intersect(marker_genes, rownames(sfe))
marker_genes

In [ ]:
scale_rows <- function(x) {
  t(base::scale(t(x)))
}

visium_scaled <- scale_rows(as.matrix(logcounts(sfe)[marker_genes, ]))
visium_scaled[, 1:5]

::: callout-important
## Exercise 3

What does a positive scaled value mean for a gene in one cell? Why should scaling be applied after normalization rather than directly to raw counts?
:::

::: {.callout-tip collapse="true"}
## Answer

A positive scaled value means that the cell has expression above the average for that gene, relative to the variation of that gene across the dataset. Scaling should be applied after normalization because raw counts are strongly affected by sequencing depth.
:::

## Optional: spatially-aware normalization

::: callout-note
## Optional discussion

In spatial transcriptomics, in addition to sequencing depth, total counts can vary because of tissue density, local capture efficiency, or morphological structures. Many of these factors are partially confounded with biology, for example with the differences observed across cell types. 

Spatially-aware normalization methods try to account for spatial structure while avoiding the removal of true spatial biological signal. This is an active area of method development, cf. the [OSTA book discussion](https://lmweber.org/OSTA/pages/ind-normalization.html) about these issues. Packages such as `SpaNorm` can be explored as optional extensions, but their basic assumption might not apply to all tissues (e.g., the cell type composition is relatively homogenous in brain so spatial smoothing might be advantageous, but it is not in our colon sample, so is smoothing a good idea?) 

For this course, we keep the required workflow simple and use `logNormCounts()`. When interpreting the downstream results, remember that spatial structure in size factors or total counts may itself be informative.
:::

::: callout-important
## Exercise 4

Look back at the spatial plot of Visium HD size factors. Would you expect a spatially-aware normalization method to remove all of that pattern? Why or why not?
:::

::: {.callout-tip collapse="true"}
## Answer

Not necessarily. Some spatial pattern in size factors may reflect technical effects, but some may reflect real tissue structure, local cell density, or biological differences. A spatially-aware method should avoid removing meaningful spatial biology while reducing unwanted technical variation.
:::

## Save normalized object

We save the normalized Visium HD object for downstream exercises.

In [ ]:
dir.create("results/day2", showWarnings = FALSE, recursive = TRUE)
alabaster.sfe::saveObject(sfe, "results/day2/02.1_sfe")

Clear your environment:

In [ ]:
#| eval: false
rm(list = ls())
gc()
.rs.restartR()

::: callout-important
**Key Takeaways:**

- `logNormCounts()` adds a `logcounts` assay and stores size factors.
- Scaling centers and rescales genes after normalization.
- Spatial structure in size factors may be technical, biological, but likely reflects both.
:::